# Tutorial: WorldQuant Brain — Bigdata.com Knowledge Graph API

Resolve company and entity names to IDs—and IDs back to names and details—so **you** can use entity and source filters in Search and Volume APIs. This tutorial shows how to use the Knowledge Graph API to **resolve company and entity names to IDs** and **look up entity and source details**. Bigdata.com Search and Volume APIs use entity IDs in filters; this API is how you map names to IDs and back.

**What you'll learn**
- Look up companies by name and get entity IDs for use in Search/Volume filters
- Resolve entity IDs to names and details (e.g. from search result `detections`)
- Run a full workflow: company lookup → Search API with entity filter → resolve IDs from chunks
- Find sources by name and filter by rank, category, or country

**Why Knowledge Graph**
- Search and Volume APIs require **entity IDs** (e.g. for `entity.any_of`); the Knowledge Graph gives you the mapping from company/entity names to IDs
- Resolve IDs returned in search chunks so you can label entities in your pipelines
- Discover source IDs and metadata for filtering or source-boost strategies

**Example use cases:** mapping company names to IDs for entity filters, resolving entity IDs from search/CoMentions results, finding source IDs for `source` or `source_boost`.

**Signals:** Entity IDs from this API are used to filter Search/Volume and to label companies in the Workflow_example signal pipeline.

**In this notebook**
1. **Company lookup** — find entity IDs by company name  
2. **Entity ID to details** — resolve one or more IDs to names and metadata  
3. **Entity discovery workflow** — company → Search with entity filter → resolve IDs from chunks  
4. **Sources** — find sources by name and filter by rank/category/country  

**Prerequisites:** `.env` file with `BRAIN_EMAIL` and `BRAIN_PASSWORD` (see `.env.example`). For access, see the challenge or [WorldQuant Brain](https://www.worldquantbrain.com/) documentation.

**Time:** about 15–20 minutes.

**Endpoint documentation:** [Knowledge Graph API reference](https://docs.bigdata.com/api-reference/knowledge-graph)

In [ ]:
from session import BrainSession
from print_helpers import (
    print_companies,
    print_entity_details
)

## Setup & Configuration

In [ ]:
# Load credentials from .env file
import os
from dotenv import load_dotenv
load_dotenv()

# WorldQuant Brain API Configuration
API_BASE_URL = "https://api.worldquantbrain.com"
EMAIL = os.getenv("BRAIN_EMAIL")
PASSWORD = os.getenv("BRAIN_PASSWORD")

# Knowledge Graph API Endpoints
KG_COMPANIES_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/knowledge-graph/companies"
KG_ENTITIES_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/knowledge-graph/entities/id"
KG_SOURCES_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/knowledge-graph/sources"

session = BrainSession(
    base_url=API_BASE_URL,
    email=EMAIL,
    password=PASSWORD
)

## Key considerations before starting

The provider APIs do not have full coverage for parameter validation—double-check your request against the [API reference](https://docs.bigdata.com/api-reference) to avoid issues from typos or invalid values.


## 1. Company lookup: name to entity ID

Search and Volume APIs filter by **entity ID** (e.g. `entity.any_of: ["D8442A"]`). To get IDs from company names, use the Knowledge Graph **Companies** endpoint: send a query string (e.g. `"Apple"`) and receive a list of matching companies with their IDs. The results are ordered by a popularity ranking that favors well-known companies, so typically the most prominent or widely-recognized companies appear first. Pick the entry that matches your intent.

**Endpoint:** [Find companies](https://docs.bigdata.com/api-reference/knowledge-graph/find-companies)

**Optional filters** (narrow results by company attributes):
- `types`: e.g. `["PUBLIC"]` (public companies), `["PRIVATE"]` (private companies)
- `countries`: ISO 3166-1 alpha-2 codes (e.g. `["US", "FR"]`)
- `sectors`: e.g. `["Technology", "Financials", "Industrials"]`

In [ ]:
company_search_payload = {
    "query": "Apple",
    # Optional: narrow by company attributes
    # "types": ["PUBLIC"],
    # "countries": ["US", "FR"],
    # "sectors": ["Technology", "Financials", "Industrials"]
}

response = session.post(KG_COMPANIES_ENDPOINT, json=company_search_payload)
data = response.json()
print_companies(response, data)

You'll see a list of matching companies with their IDs—pick the entry that matches your intent. Try changing the query (e.g. `"Microsoft"`, `"Tesla"`), using additional filters, and re-running the cell above to find other company IDs.

You can also look up companies directly using standard identifiers such as ISIN, CUSIP, SEDOL, or TICKER.
The Knowledge Graph API includes dedicated endpoints for these identifier lookups:
- `/knowledge-graph/companies/isin` — Find a company using its ISIN
- `/knowledge-graph/companies/cusip` — Find a company using its CUSIP
- `/knowledge-graph/companies/listing` — Find a company by ticker or listing code

For example, providing an ISIN or ticker will return the corresponding company entity.

## 2. From Entity ID to details

Next, we resolve **entity IDs to names and metadata**—useful when you have IDs from Search or CoMentions results. When you have a list of entity IDs—such as those returned from Search or CoMentions results (`detections` in chunks)—you can resolve them to names and metadata using the **Entities by ID** endpoint. Simply send up to 100 RavenPack IDs per request and receive a mapping of ID to entity details (name, type, etc.), which is especially useful for labeling entities in your pipelines.

**Endpoint:** [Get entities by ID](https://docs.bigdata.com/api-reference/companies/find-by-details)

In [ ]:
entity_lookup_payload = {
    "values": ["D8442A"]
}

response = session.post(KG_ENTITIES_ENDPOINT, json=entity_lookup_payload)
data = response.json()
print(data)

## 3. Entity discovery workflow

We now tie the Knowledge Graph and Search APIs together in one workflow.

**Why this workflow?** When you search with an entity filter, each chunk includes `detections`: entity IDs mentioned in that text. To see **who** or **what** those are—and to discover which companies, people, or topics are co-mentioned with your focal entity—you resolve those IDs with the Knowledge Graph. This workflow shows that full loop: company name → filtered search → entity IDs from chunks → resolved names.

Steps:
1. **Look up** a company (e.g. Microsoft) to get its entity ID  
2. **Search** with that entity filter (Search API)  
3. **Extract** entity IDs from the first chunk’s `detections`  
4. **Resolve** those IDs to names via the Knowledge Graph  

You can reuse this pattern whenever you need to go from company name → filtered search → labelled entities.

In [ ]:
# Find Microsoft Entity ID
# For well-known companies with correct spelling, the first result is typically correct
company_query = {"query": "Microsoft"}
response = session.post(KG_COMPANIES_ENDPOINT, json=company_query)
companies = response.json()['results']

microsoft_id = companies[0]['id']
microsoft_name = companies[0]['name']

print(f"Found: {microsoft_name} (ID: {microsoft_id})")

In [ ]:
# Example: Search with Microsoft Filter (using Search API)
# Note: This requires the Search API endpoint, which is shown here for completeness
# In a real workflow, you would use this entity ID with the Search API

SEARCH_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/search"
TEXT = "Global semiconductor shortage impacts"
START_DATE = "2021-01-01"
END_DATE = "2021-12-30"

search_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z"
            },
            "entity": {"any_of": [microsoft_id]}
        },
        "ranking_params": {"freshness_boost": 0},
        "max_chunks": 50
    }
}

response = session.post(SEARCH_ENDPOINT, json=search_query)
search_results = response.json()

print(f"Found {len(search_results.get('results', []))} documents")

In [ ]:
# Extract Entity IDs from First Chunk
if search_results.get('results'):
    first_chunk = search_results['results'][0]['chunks'][0]
    
    entity_ids = [
        d['id'] for d in first_chunk['detections'] 
        if d['type'] == 'entity'
    ]
    
    print(f"Found {len(entity_ids)} entity IDs in first chunk")
    print(f"Sample IDs: {entity_ids[:5]}")
else:
    print("No search results available")
    entity_ids = []

In [ ]:
# Resolve Entity IDs to Names
if entity_ids:
    response = session.post(KG_ENTITIES_ENDPOINT, json={"values": entity_ids})
    results = response.json()['results']
    
    # Results is a dict: entity_id -> entity_object
    entity_map = {entity_id: entity['name'] for entity_id, entity in results.items()}
    
    print(f"Resolved {len(entity_map)} entities:")
    for entity_id, name in list(entity_map.items())[:10]:
        print(f"  {entity_id}: {name}")
else:
    print("No entity IDs to resolve")

## 4. Sources: find source IDs and metadata

Finally, we look at **sources**—how to find source IDs by name and filter by rank, category, or country. The Search API returns documents with a `source` object (id, name, rank). To find source IDs by name or to filter sources by quality/category/country, use the Knowledge Graph **Sources** endpoint. Handy when you want to restrict results to certain publishers or align with `source_boost` strategies.

**Endpoint:** [Find sources](https://docs.bigdata.com/api-reference/knowledge-graph/find-sources)

**Available filters**
- `ranks`: e.g. `["RANK_1", "RANK_2"]` (RANK_1 = highest quality)
- `categories`: `["news", "transcripts", "research", "podcasts", "filings", "expert_interviews"]`
- `countries`: ISO 3166-1 alpha-2 codes (e.g. `["US", "GB"]`)
- `packages`: data packages (e.g. `["sec_filings"]`)

In [ ]:
# Example: Find Reuters sources with high quality rank
source_query = {
    "query": "Reuters",
    "ranks": ["RANK_1", "RANK_2"],
    "categories": ["news"]
}

response = session.post(KG_SOURCES_ENDPOINT, json=source_query)
sources = response.json()['results']

# Get first source (most relevant)
if sources:
    reuters_source_id = sources[0]['id']
    reuters_source_name = sources[0]['name']
    reuters_source_rank = sources[0]['rank']
    print(f"Reuters source ID: {reuters_source_id}, name: {reuters_source_name}, rank: {reuters_source_rank}")
else:
    print("No sources found")

## Summary and next steps

**What we covered**
- Company lookup by name to get entity IDs for Search/Volume filters
- Resolving entity IDs to names and details (e.g. from search chunk `detections`)
- End-to-end workflow: company → Search with entity filter → resolve IDs from chunks
- Finding sources by name and filtering by rank, category, or country

**Next steps**
- **Search API** ([Search_API](../Search_API/)) — semantic search with entity, time, and sentiment filters
- **Volume API** ([Volume_API](../Volume_API/)) — document and chunk volume over time
- **CoMentions API** ([CoMentions_API](../CoMentions_API/)) — entity co-occurrence; use Knowledge Graph to resolve IDs to names
- **Full API reference:** [docs.bigdata.com](https://docs.bigdata.com)

You can experiment by changing the company query (e.g. `"Microsoft"`, `"Tesla"`) or the source query (e.g. `"Reuters"`) and re-running the notebook to explore different lookups.